# Call the live model server

The Concrete Evidence model server runs on Render: **https://concrete-evidence-model.onrender.com**.
This notebook sends it one photo of a column face and reads the answer, the same call the Cloudflare app makes.

- The free plan sleeps after 15 minutes without calls. The first call after that takes about a minute while it wakes; later calls take about 20 seconds.
- Model now: the skewer stand-in (`iaac-maai/concrete-evidence-stand-in`), so the faults below are about skewers, not real rebar. From Mon 23 Nov: the rebar model.
- No photo is saved in this notebook: only numbers and the fix list.

In [1]:
# pip install gradio_client
import base64, json, os, time
from pathlib import Path
from gradio_client import Client

URL = "https://concrete-evidence-model.onrender.com"
PHOTO = Path(os.environ.get("CE_PHOTO", Path.home() / "concrete-evidence/code/skewer_yolo_notebook/photos/extra_01.jpg"))
print(PHOTO.name, PHOTO.stat().st_size // 1000, "kB")

extra_01.jpg 62 kB


## 1. The endpoint

One endpoint, `/predict`: a base64 photo, the bar bending schedule as JSON (empty = the example schedule), the zone (`end` or `middle`) and the face. It answers the check, every piece found with its outline, and the photo with faces blurred and outlines drawn.

In [2]:
t = time.time()
client = Client(URL, verbose=False)   # wakes the server if it sleeps
print(f"connected in {time.time() - t:.0f} s")
client.view_api(print_info=False, return_format="dict")["named_endpoints"].keys()

connected in 1 s


dict_keys(['/predict'])

## 2. One photo

In [3]:
image_b64 = base64.b64encode(PHOTO.read_bytes()).decode()
t = time.time()
result = client.predict(image_b64, "", "end", "A", api_name="/predict")
print(f"answered in {time.time() - t:.1f} s")
annotated = result.pop("annotated_jpg_base64", "")  # the checked photo: not shown or saved here
print("keys:", sorted(result))
print("model:", result["model"])

answered in 14.0 s
keys: ['check', 'counts', 'depth', 'detections', 'faces_blurred', 'markers_seen', 'mm_per_px', 'model']
model: {'weights': 'best.onnx', 'rename': {'skewer': 'bar'}, 'depth_filter': False}


## 3. What it found

In [4]:
check = result["check"]
print("pieces found:", len(result["detections"]), "| by class:", result["counts"])
print("markers seen:", result["markers_seen"], "| mm per pixel:", result["mm_per_px"])
print("faces blurred:", result["faces_blurred"])
print("faults:", check["faults"])
for fix in check.get("fixes", [])[:8]:
    print(" -", fix)

pieces found: 72 | by class: {'bar': 72}
markers seen: [] | mm per pixel: None
faces blurred: 0
faults: 37
 - Column C1, face A: Check the 69 extra bar(s): 72 seen, 3 scheduled.
 - Column C1, face A: Move bars 1 and 10 to 155 mm apart (now 9 mm).
 - Column C1, face A: Move bars 10 and 17 to 155 mm apart (now 8.9 mm).
 - Column C1, face A: Move bars 17 and 31 to 155 mm apart (now 3.4 mm).
 - Column C1, face A: Move bars 31 and 35 to 155 mm apart (now 3.6 mm).
 - Column C1, face A: Move bars 14 and 35 to 155 mm apart (now 7.3 mm).
 - Column C1, face A: Move bars 14 and 3 to 155 mm apart (now 57.1 mm).
 - Column C1, face A: Move bars 13 and 3 to 155 mm apart (now 7 mm).


In [5]:
failed = [c for c in check.get("checks", []) if c.get("status") == "fail"]
passed = [c for c in check.get("checks", []) if c.get("status") == "pass"]
print(len(passed), "rules pass,", len(failed), "fail")
print(json.dumps(failed[:2], indent=1)[:1500])

0 rules pass, 37 fail
[
 {
  "check": "bar_count",
  "status": "fail",
  "detail": "69 bar(s) extra: 72 seen, 3 scheduled",
  "measured": 72,
  "expected": 3,
  "track_ids": []
 },
 {
  "check": "spacing",
  "status": "fail",
  "detail": "gap of 9.0 mm, scheduled 155 \u00b1 10 mm",
  "measured": 9.0,
  "expected": 155,
  "track_ids": [
   1,
   10
  ]
 }
]


## What the app does with this

The Cloudflare app (https://concrete-evidence.pages.dev, team password) sends the same call from its server code,
stores the checked photo and the result, writes one row to the database, and shows the fix list.
A person then confirms or rejects each fault before the pour: the model only suggests.